# Data Cleaning LOTR Flora

## 1. Basic Setup

This notebook assumes that the user has followed the steps outlined in the `README.md` procedure. Where there is cause for discrepancy from the original `flora.csv`, I have made a note of which lines to change. You really should only need to change either the code dealing with plant name formatting or chapter title formatting, because those will slightly differ from person to person when recording values in the data.

Of course, if the user is not interested in recreating the dataset from scratch and would just like to see the methodology of the data cleaning process, then they don't need to change any lines of code in this file. Just run the `flora_raw.csv` through this script and you should end up with a file exactly like `flora.csv`. 

In [102]:
# import libraries (we only need one here lol)
import pandas as pd

In [103]:
# read in the csv into a df
# replace the file name with whatever you named your own csv file
# if you're just following along, don't edit this code block
flora = pd.read_csv("flora_raw.csv")

In [104]:
# fill in empty cells so we're not working with nulls
flora = flora.fillna({"real": "yes", "name": "no"})

In [105]:
# split chapter;frequency pairs within the fotr, tt, rotk columns so that each cell is a list of chapter;frequency string pairs
books = ["fotr", "tt", "rotk"]

for book in books:
    flora[book] = flora[book].fillna("")
    flora[book] = flora[book].str.split(",")

## 2. Anti-Pivot Tables

Ultimately, we want the final df to have a unique `plant` + `chapter` combination for every row. We'll need to first melt the df so that the columns `fotr`, `tt`, and `rotk` are themselves values in a new `book` column, and the columns' old values are now called `chapter`.

In [106]:
# introduce two new columns, "book" and "chapter"
keep_cols = ["plant", "real", "name"]
flora_melted = pd.melt(flora, id_vars=keep_cols, value_vars=books, var_name="book", value_name="chapter")

In [107]:
# split up "chapters" column so each chapter;frequency pair gets their own row
flora_exploded = flora_melted.explode("chapter")
flora_exploded = flora_exploded[flora_exploded["chapter"] != ""] # drop rows with no chapter values

# split up each chapter;frequency pair so the chapter and frequency each have their own columns
flora_exploded["chapter"] = flora_exploded["chapter"].str.split(";")
flora_exploded["count"] = flora_exploded["chapter"].str[1]
flora_exploded["chapter"] = flora_exploded["chapter"].str[0]

flora_exploded = flora_exploded.reset_index().drop("index", axis=1)

## 3. Some More Standardization

In [108]:
# some definitions and mappings to properly capitalize volume and chapter titles
# if your chapter titles differ slightly from mine, you should edit book#_chapters' keys with your own titles
# leave the values of the dictionaries unchanged, as they are the "properly formatted" chapter titles
book_map = {"fotr": "The Fellowship of the Ring", "tt": "The Two Towers", "rotk": "The Return of the King"}

book1_chapters = {"a long-expected party": "A Long-expected Party",
               "the shadow of the past": "The Shadow of the Past",
               "three is company": "Three is Company",
               "a short cut to mushrooms": "A Short Cut to Mushrooms",
               "a conspiracy unmasked": "A Conspiracy Unmasked",
               "the old forest": "The Old Forest",
               "in the house of tom bombadil": "In the House of Tom Bombadil",
               "fog on the barrow-downs": "Fog on the Barrow-downs",
               "at the sign of the prancing pony": "At the Sign of The Prancing Pony",
               "strider": "Strider",
               "a knife in the dark": "A Knife in the Dark",
               "flight to the ford": "Flight to the Ford"}

book2_chapters = {"many meetings": "Many Meetings",
               "the council of elrond": "The Council of Elrond",
               "the ring goes south": "The Ring Goes South",
               "a journey in the dark": "A Journey in the Dark",
               "the bridge of khazad-dûm": "The Bridge of Khazad-dûm",
               "lothlórien": "Lothlórien",
               "the mirror of galadriel": "The Mirror of Galadriel",
               "farewell to lórien": "Farewell to Lórien",
               "the great river": "The Great River",
               "the breaking of the fellowship": "The Breaking of the Fellowship"}

book3_chapters = {"the departure of boromir": "The Departure of Boromir",
               "the riders of rohan": "The Riders of Rohan",
               "the uruk-hai": "The Uruk-hai",
               "treebeard": "Treebeard",
               "the white rider": "The White Rider",
               "the king of the golden hall": "The King of the Golden Hall",
               "helm's deep": "Helm's Deep",
               "the road to isengard": "The Road to Isengard",
               "flotsam and jetsam": "Flotsam and Jetsam",
               "the voice of saruman": "The Voice of Saruman",
               "the palantír": "The Palantír"}

book4_chapters = {"the taming of sméagol": "The Taming of Sméagol",
               "the passage of the marshes": "The Passage of the Marshes",
               "the black gate is closed": "The Black Gate is Closed",
               "of herbs and stewed rabbit": "Of Herbs and Stewed Rabbit",
               "the window on the west": "The Window on the West",
               "the forbidden pool": "The Forbidden Pool",
               "journey to the cross-roads": "Journey to the Cross-Roads",
               "the stairs of cirith ungol": "The Stairs of Cirith Ungol",
               "shelob's lair": "Shelob's Lair",
               "the choices of master samwise": "The Choices of Master Samwise"}

book5_chapters = {"minas tirith": "Minas Tirith",
               "the passing of the grey company": "The Passing of the Grey Company",
               "the muster of rohan": "The Muster of Rohan",
               "the siege of gondor": "The Siege of Gondor",
               "the ride of the rohirrim": "The Ride of the Rohirrim",
               "the battle of the pelennor fields": "The Battle of the Pelennor Fields",
               "the pyre of denethor": "The Pyre of Denethor",
               "the houses of healing": "The Houses of Healing",
               "the last debate": "The Last Debate",
               "the black gate opens": "The Black Gate Opens"}

book6_chapters = {"the tower of cirith ungol": "The Tower of Cirith Ungol",
               "the land of shadow": "The Land of Shadow",
               "mount doom": "Mount Doom",
               "the field of cormallen": "The Field of Cormallen",
               "the steward and the king": "The Steward and the King",
               "many partings": "Many Partings",
               "homeward bound": "Homeward Bound",
               "the scouring of the shire": "The Scouring of the Shire",
               "the grey havens": "The Grey Havens"
}

chapters_list = [book1_chapters, book2_chapters, book3_chapters, book4_chapters, book5_chapters, book6_chapters]
chapters_map = dict(item for d in chapters_list for item in d.items())

In [109]:
# this csv will facilitate with the natural ordering inherent in chapter and book numbers
chapters = pd.read_csv("chapters.csv")

In [110]:
# if you'd like to change any plant names, define the mapping here (original -> new)
# change plant_terms as you see fit
plant_terms = {
    "bay-leaf": "bay",
}

In [111]:
# if you'd like to change any column names, define the mapping here (original -> new)
# change flora_columns as you see fit
flora_columns = {
    "plant": "Plant",
    "real": "Real?",
    "chapter": "Chapter",
    "count": "Count",
    "volume": "Volume",
    "book": "Book",
    "order": "Order"
}

## 4. Putting It All Together

In [112]:
# clean up formatting/capitalization
flora_exploded["book"] = flora_exploded["book"].map(book_map)
flora_exploded["chapter"] = flora_exploded["chapter"].map(chapters_map)
flora_exploded["count"] = flora_exploded["count"].astype("int64")

In [113]:
# merge flora_exploded with chapters to introduce inherent order with the chapters and books
flora = flora_exploded.merge(chapters, how="right", left_on="chapter", right_on="chapter").drop("book_x", axis=1)

# clean up formatting
flora = flora.rename(columns={"book_y": "book"})
flora = flora.rename(columns=flora_columns)

flora["Plant"] = flora["Plant"].replace(plant_terms)

flora["Count"] = flora["Count"].fillna(0)
flora["Count"] = flora["Count"].astype("int64")

In [114]:
# export to final cleaned flora.csv 
# uncomment below to export 
# flora.to_csv("flora.csv", index=False)